In [ ]:
import asyncio
import hashlib
import json
from pathlib import Path

from IPython.display import display

import xyg
from test_graphforge import _base
from xyg import _native

native_path = Path(_native._lib._name).resolve()
assert native_path == Path(xyg.__file__).resolve().parents[2] / "target/release/libxyg_core.so"
native = {
    "path": str(native_path),
    "sha256": hashlib.sha256(native_path.read_bytes()).hexdigest(),
    "abi": int(_native._lib.xyg_abi_version()),
}
composition = xyg.compose_graphforge(base=_base("property"))
clicks, hovers, views, selections = [], [], [], []
chart = xyg.graphforge_chart(
    composition,
    title="Progressive property graph",
    on_click=clicks.append,
    on_hover=hovers.append,
    on_view_change=views.append,
    on_select=lambda selection: selections.append(selection.rows()),
)
widget = chart.show(display="widget")
assert chart._figure is None
display(widget)
other_clicks = []
other = xyg.graphforge_chart(
    composition, theme="dark", title="Independent graph", on_click=other_clicks.append
).widget()
display(other)

In [ ]:
assert asyncio.get_running_loop().is_running()
assert clicks and hovers and views, (clicks, hovers, views)
assert clicks[0]["id"] in composition.nodes.uuid
assert clicks[0]["graphforge"] == [composition.identify("node", 0)]
assert not other_clicks
print(
    "XYG graph kernel responsive",
    json.dumps(
        {
            "clicks": clicks,
            "native": native,
            "otherClicks": len(other_clicks),
            "hovers": len(hovers),
            "views": len(views),
            "selections": len(selections),
        }
    ),
)

In [ ]:
chosen_uuids = [composition.nodes.uuid[0], composition.edges.uuid[0]]
widget.select(uuids=chosen_uuids)
print(
    "XYG graph Python rows selected",
    json.dumps(
        {
            "requested": chosen_uuids,
            "identities": [composition.identify("node", 0), composition.identify("edge", 0)],
        }
    ),
)

In [ ]:
widget.reset_view()
print("XYG graph reset")

In [ ]:
final_pointer_before = len(clicks)
print("XYG graph pointer baseline", final_pointer_before)

In [ ]:
async with asyncio.timeout(5):
    while len(clicks) <= final_pointer_before:
        await asyncio.sleep(0.01)
assert clicks[-1]["id"] == composition.nodes.uuid[0]
assert clicks[-1]["graphforge"] == [composition.identify("node", 0)]
assert not other_clicks
widget.close()
other.close()
print(
    "XYG graph disposed",
    json.dumps(
        {
            "before": final_pointer_before,
            "after": len(clicks),
            "id": clicks[-1]["id"],
            "expectedId": composition.nodes.uuid[0],
        }
    ),
)

In [ ]:
import base64

identity = widget._figure._notebook_graph_identity
meta = widget._figure._graph_meta[0]
edge_identity = widget._figure._graph_edge_identity[meta["edge_trace"]]
identity_fixture = {
    "composition": base64.b64encode(composition.bytes).decode("ascii"),
    "positions": {
        "uuid": list(composition.nodes.uuid),
        "x": identity._positions[0].tolist(),
        "y": identity._positions[1].tolist(),
    },
    "edgeTrace": meta["edge_trace"],
    "nodeTrace": meta["node_trace"],
    "edgeSources": [
        edge_identity.members_of(int(i)).tolist() for i in edge_identity.render_edge_index
    ],
    "expected": [composition.identify("edge", i) for i in range(composition.edges.count)],
}
connected_y = [float(identity._positions[1][i]) for i in range(4)]
lo, hi = min(connected_y), max(connected_y)
widget.set_view({"y": [lo - 0.5, hi + 0.5]}, animate=False)
print("XYG identity fixture", json.dumps(identity_fixture))

In [ ]:
edge_clicks = [r for r in clicks if r.get("graphforge") and r["graphforge"][0]["kind"] == "edge"]
assert {r["graphforge"][0]["uuid"] for r in edge_clicks} >= set(composition.edges.uuid), edge_clicks
assert not other_clicks
assert identity.session._handle is None
print(
    "XYG routed identity",
    json.dumps({"fixture": identity_fixture, "clicks": edge_clicks, "identityReleased": True}),
)

In [ ]:
from test_graphforge import _compose

layered_composition = _compose("pagerank", result_id="notebook-rank")
layered_clicks, layered_hovers = [], []
layered_widget = xyg.graphforge_chart(
    layered_composition,
    title="Layered identity",
    on_click=layered_clicks.append,
    on_hover=layered_hovers.append,
).widget()
display(layered_widget)

In [ ]:
async with asyncio.timeout(5):
    while not layered_clicks:
        await asyncio.sleep(0.01)
expected_layered = layered_composition.identify("node", 0)
assert layered_clicks[-1]["graphforge"] == [
    {k: expected_layered[k] for k in ("kind", "uuid", "index", "layers")}
]
assert expected_layered["layers"] == [{"layer": 0, "result_id": "notebook-rank", "row": 0}]
layered_identity = layered_widget._figure._notebook_graph_identity
layered_fixture = {
    "composition": base64.b64encode(layered_composition.bytes).decode("ascii"),
    "positions": {
        "uuid": list(layered_composition.nodes.uuid),
        "x": layered_identity._positions[0].tolist(),
        "y": layered_identity._positions[1].tolist(),
    },
    "expected": expected_layered,
    "click": layered_clicks[-1],
}
layered_session = layered_identity.session
layered_widget.close()
assert layered_session._handle is None
layered_fixture["identityReleased"] = True
print("XYG layered identity", json.dumps(layered_fixture))